# 02 - Data Modeling

Load raw datasets into DuckDB and build the analytical model.

In [1]:
import duckdb
from pathlib import Path

In [2]:
conn = duckdb.connect('../customer_analytics.duckdb')

In [3]:
DATA_PATH = '../data/raw'

## Raw Tables

In [4]:
conn.execute(f''' 
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM read_csv_auto('{DATA_PATH}/olist_customers_dataset.csv');
''')

In [5]:
conn.execute(f''' 
CREATE OR REPLACE TABLE orders AS
SELECT *
FROM read_csv_auto('{DATA_PATH}/olist_orders_dataset.csv');
''')

In [6]:
conn.execute(f''' 
CREATE OR REPLACE TABLE order_items AS
SELECT *
FROM read_csv_auto('{DATA_PATH}/olist_order_items_dataset.csv');
''')

In [7]:
conn.execute(f''' 
CREATE OR REPLACE TABLE products AS
SELECT *
FROM read_csv_auto('{DATA_PATH}/olist_products_dataset.csv');
''')

In [8]:
conn.execute(f''' 
CREATE OR REPLACE TABLE payments AS
SELECT *
FROM read_csv_auto('{DATA_PATH}/olist_order_payments_dataset.csv');
''')

### Validation

In [9]:
conn.execute('SHOW TABLES').fetchdf()

,name
0,customers
1,dim_customer
2,dim_date
3,dim_product
4,fct_sales
5,order_items
6,orders
7,payments
8,products
9,silver_customers


In [10]:
conn.execute('''
SELECT 'customers' as table_name, count(*) as rows FROM customers
UNION ALL
SELECT 'orders', count(*) FROM orders
UNION ALL
SELECT 'order_items', count(*) FROM order_items
UNION ALL
SELECT 'products', count(*) FROM products
UNION ALL
SELECT 'payments', count(*) FROM payments
''').fetchdf()

,table_name,rows
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,payments,103886


## Silver Layer

In [11]:
silver_files = [
    "../sql/00_silver_customers.sql",
    "../sql/00_silver_orders.sql",
    "../sql/00_silver_order_items.sql",
    "../sql/00_silver_products.sql",
    "../sql/00_silver_payments.sql"
]

for file in silver_files:

    with open(file, "r") as f:
        conn.execute(f.read())

print("Silver layer created successfully!")

Silver layer created successfully!


### Validation

In [12]:
conn.execute("""
SHOW TABLES
""").fetchdf()

,name
0,customers
1,dim_customer
2,dim_date
3,dim_product
4,fct_sales
5,order_items
6,orders
7,payments
8,products
9,silver_customers


### Export to Silver

In [13]:
silver_tables = [
    "silver_customers",
    "silver_orders",
    "silver_order_items",
    "silver_products",
    "silver_payments"
]

for table in silver_tables:

    conn.execute(f"""
        COPY {table}
        TO '../data/silver/{table}.parquet'
        (FORMAT PARQUET)
    """)

print("Silver exported!")

Silver exported!


## Gold Layer

In [14]:
gold_files = [
    "../sql/01_create_fct_sales.sql",
    "../sql/02_create_dim_customer.sql",
    "../sql/03_create_dim_product.sql",
    "../sql/04_create_dim_date.sql"
]

for file in gold_files:

    with open(file, "r") as f:
        conn.execute(f.read())

print("Gold layer created successfully!")

Gold layer created successfully!


In [15]:
gold_tables = [
    "fct_sales",
    "dim_customer",
    "dim_product",
    "dim_date"
]

for table in gold_tables:

    conn.execute(f"""
        COPY {table}
        TO '../data/gold/{table}.parquet'
        (FORMAT PARQUET)
    """)

print("Gold exported!")

Gold exported!


## DuckDB validation

In [17]:
conn.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
""").fetchdf()

,table_name
0,customers
1,dim_customer
2,dim_date
3,dim_product
4,fct_sales
5,orders
6,order_items
7,payments
8,products
9,silver_customers
